# Обучение RNN

In [1]:
from tqdm import tqdm

import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader

In [2]:
# do not change the code in the block below
# __________start of block__________
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print('{} device is available'.format(device))
# __________end of block__________

# Для воспроизводимости результатов фиксируем seed
seed = 43

if torch.cuda.is_available() and device.type == 'cuda':
    np.random.seed(seed)
     # Если используете CUDA, устанавливаем seed для GPU                
    torch.cuda.manual_seed_all(seed)
    # Для полной воспроизводимости также можно зафиксировать алгоритмы cuDNN
    torch.backends.cudnn.deterministic = True   
    torch.backends.cudnn.benchmark = False
else:
    # Устанавливаем seed для CPU генератора случайных чисел PyTorch
    np.random.seed(seed)
    torch.manual_seed(seed)


# Несколько оптимизаций которые могут ускорить процесс обучения
if torch.backends.cudnn.is_available():
    print(f'Using cudnn version: {torch.backends.cudnn.version()}')
    if device.type == 'cuda':
        torch.backends.cudnn.enabled = True
        torch.backends.cudnn.benchmark = True

cuda device is available
Using cudnn version: 90100


## Загрузка данных

In [3]:
class OneginDataset(Dataset):
    def __init__(self, path, seq_length, transform=None):

        # Откроем файл и считаем все строки текста
        with open('onegin.txt', 'r') as iofile:
            self.text = iofile.readlines()
    
        self.text = "".join([x.replace('\t\t', '').lower() for x in self.text])


        # Создадим словари для отображения символов в токены и обратно
        self.tokens = sorted(set(self.text)) + ['<sos>']
        self.num_tokens = len(self.tokens)
        self.token_to_idx = {x: idx for idx, x in enumerate(self.tokens)}
        self.idx_to_token = {idx: x for idx, x in enumerate(self.tokens)}

        # Преобразуем текст в последовательность токенов
        self.text_encoded = torch.tensor([self.token_to_idx[x] for x in self.text])
        self.text_encoded = torch.split(self.text_encoded, seq_length)
        self.text_encoded = tuple(torch.cat((t, torch.ones(seq_length - t.size(0), dtype=t.dtype))) if t.size(0) < seq_length else t for t in self.text_encoded) # Говно паддинг
        self.text_encoded = tuple(torch.cat((torch.tensor([83]), t)) for t in self.text_encoded)


    def __len__(self):
        return len(self.text_encoded)

    def __getitem__(self, idx):
        return self.text_encoded[idx]

Проверка датасета и загрузчика данных

In [4]:
onegin_dataset = OneginDataset('onegin.txt', seq_length=100)
dataloader = DataLoader(onegin_dataset, batch_size=256, shuffle=True, num_workers=8)

num_tokens = onegin_dataset.num_tokens

## Определение модели

In [ ]:
import torch
import torch.nn as nn
from torch.autograd import Variable

class RNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, n_layers=1):
        super(RNN, self).__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.n_layers = n_layers
        
        self.encoder = nn.Embedding(input_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, n_layers)
        self.decoder = nn.Linear(hidden_size, output_size)
    
    def forward(self, input, hidden):
        input = self.encoder(input.view(1, -1))
        output, hidden = self.rnn(input.view(1, 1, -1), hidden)
        output = self.decoder(output.view(1, -1))
        return output, hidden

    def init_hidden(self):
        return Variable(torch.zeros(self.n_layers, 1, self.hidden_size))
    


model = RNN(input_size=num_tokens, hidden_size=512, output_size=num_tokens, n_layers=3)

Функция обучения нейронной сети:

In [1]:
from torch.optim.lr_scheduler import LinearLR

def train(model, dataloader, num_epochs, criterion, optimizer):
    
    all_losses = []

    model.to(device)
    model.train()

    # Итерируемся по эпохам
    for epoch in tqdm(range(num_epochs)):
        
        total_loss = 0
                
        # Итерируемся по батчам
        for batch in dataloader:
            batch = batch.to(device)

            optimizer.zero_grad()

            hidden = model.init_hidden().to(device)
            model.zero_grad()

            loss = 0

            outputs = []
            targets = []

            # Для каждого примера в батче
            for example in batch:

                input = example[:-1]
                target = example[1:]


                # Для каждого символа в примере
                for chr_num in range(len(input)):
                    output, hidden = model(input[chr_num], hidden)
                    
                    # Накапливаем всю строку
                    outputs.append(output)
                    targets.append(target[chr_num].unsqueeze(0))

                    # loss += criterion(output, target[chr_num].unsqueeze(0))
                    
            
            outputs = torch.cat(outputs, dim=0)
            targets = torch.cat(targets, dim=0)
            loss = criterion(outputs, targets)
            
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        # Обновление планировщика после каждой эпохи
        # Корректировка скорости обучения на основе полной оценки потерь на всем наборе данных
        scheduler.step()

        print(f'Epoch {epoch+1}/{num_epochs}, Loss: {total_loss/len(dataloader)}, lr: {scheduler.get_last_lr()[0]:.4e}')




leraning_rate = 0.001
total_epoch = 40
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=leraning_rate)
# Линейно уменьшаем скорость обучения
scheduler = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=1.0, end_factor=0.001, total_iters=total_epoch)

train(model, dataloader, num_epochs=total_epoch, criterion=criterion, optimizer=optimizer)

NameError: name 'nn' is not defined

In [7]:
token_to_idx = onegin_dataset.token_to_idx
idx_to_token = onegin_dataset.idx_to_token


def generate_sample(char_rnn, seed_phrase=None, max_length=200, temperature=1.0, device=device):
    '''
    The function generates text given a phrase of length at least SEQ_LENGTH.
    :param seed_phrase: prefix characters. The RNN is asked to continue the phrase
    :param max_length: maximum output length, including seed_phrase
    :param temperature: coefficient for sampling.  higher temperature produces more chaotic outputs,
                         smaller temperature converges to the single most likely output
    '''

    if seed_phrase is not None:
        x_sequence = [token_to_idx['<sos>']] + [token_to_idx[token] for token in seed_phrase]
    else: 
        x_sequence = [token_to_idx['<sos>']]

    x_sequence = torch.tensor([x_sequence], dtype=torch.int64).to(device)

    # подаем начальную фразу, если таковая имеется
    hidden = char_rnn.init_hidden().to(device)
    for i in range(len(seed_phrase) if seed_phrase is not None else 0):
        output, hidden = char_rnn(x_sequence[:, i], hidden)

   # начинаем генерировать
    for i in range(len(seed_phrase) if seed_phrase is not None else 0, max_length):
        output, hidden = char_rnn(x_sequence[:, -1], hidden)
        
        # выборка из модели в виде мультиномиального распределения
        output_dist = output.data.view(-1).div(temperature).exp()
        top_i = torch.multinomial(output_dist, 1)[0]

        # Добавляем предсказанный символ в строку и используем как следующий входной символ
        predicted_char = idx_to_token[top_i.item()]
        x_sequence = torch.cat((x_sequence, torch.tensor([[top_i]], dtype=torch.int64).to(device)), dim=1)

        if predicted_char == '<eos>':
            break

    return ''.join([idx_to_token[ix] for ix in x_sequence.cpu().data.numpy()[0]])

In [8]:
print(generate_sample(model, ' мой дядя самых честных правил', max_length=500, temperature=0.8))

<sos> мой дядя самых честных правила нак кдо не семогнивей колсьтей ласе де, ругось иверна, вхочио в е
енне дарит
шам люкод€оф
дезцаы мувлец

чтвать осала.
«ван ет пломюттил и не мерет берите.
)
а говливо скал,
на москоль 
в мода уже е зален,
ося слене дночет утремал ной бы дорил не пых вовомана,
и скужной илим вете люш— в резстидаким мустим дуж
станнодывы вседим енекала



xvxv
танух даватдай ноговополы дроз мольмит.
ек, гой неззведне
и в следет,
чат кавье нем
и и мизни проз и стола
ник пеосни… сто 


## Сдача задания

In [18]:
seed_phrase = ' мой дядя самых честных правил'

In [19]:
generated_phrases = [
    generate_sample(
        model,
        ' мой дядя самых честных правил',
        max_length=500,
        temperature=1.
    ).replace('<sos>', '')
    for _ in range(10)
]

generated_phrases

[' мой дядя самых честных правилновили;\nпередом,\nнаклажале, захого,\nмлговь шезлу, плеч.\nкоча сем дунем,\nсадя поскожрогда, можали,\nдаздним чету\nдакага презвальки мни\nи длю.\nвдровое.\nподу длазон.\nв това мом свух ды, очтом обиньбовить оди серды власвить,\nнелаке-заровет,\nпредареньем ертавется не суфну\nон внотрына помрогогиго баё) блоеной\nи дрегой.\n\n\n\nxiт\n\nонен сумал, нас виз-мов был, блого…\nсек стилаже,\nодн.\nи йале.\nна вдушто мна нигда вокралсе,\nустивухный жил этьяковь ит отрам вдланый сечах, и говост',
 ' мой дядя самых честных правилаелнычный\nвужа,\nкак оны длядик.\nуж молабы у копосты, споk лену гричен но лугое мудиня эли бый силинибыва прев, жвечко, шернотом\nевлем при сечю терой про чавтой ведих тотьять,\nни всерте онь ба жали стрешь накоратитрый.\nпойдадарыть ынестей коввлкаття. \n\n\nxxvii\n\nвдель,\nне надени миц и всё дуто,\nнеже тенья,\nбрюз пнадоречных дружстапешь из гостногы свени хлядкись и тот кию ра встраская сревый оскам не бладтеня, нустала не дк

In [ ]:
# do not change the code in the block below
# __________start of block__________

import json
if 'generated_phrases' not in locals():
    raise ValueError("Please, save generated phrases to `generated_phrases` variable")

for phrase in generated_phrases:

    if not isinstance(phrase, str):
        raise ValueError("The generated phrase should be a string")

    if len(phrase) != 500:
        raise ValueError("The `generated_phrase` length should be equal to 500")

    assert all([x in set(tokens) for x in set(list(phrase))]), 'Unknown tokens detected, check your submission!'
    

submission_dict = {
    'token_to_idx': token_to_idx,
    'generated_phrases': generated_phrases
}

with open('submission_dict.json', 'w') as iofile:
    json.dump(submission_dict, iofile)
print('File saved to `submission_dict.json`')
# __________end of block__________